# U10-01 · Matplotlib, color y Seaborn

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 10, apartados 10.2 a 10.5*

Ejecuta cada celda con **Mayús + Intro**. Para que funcione en el navegador, la flota simulada tiene **50 máquinas** en lugar de 200: las máquinas 0-24 son las del sensor descalibrado (en el libro, 0-99). Las figuras se parecerán a las del libro, pero no serán idénticas.

In [ ]:
import os, sys
if sys.platform == "emscripten":
    # En el navegador, DuckDB no lee la carpeta de JupyterLite:
    # se trabaja en una carpeta temporal en memoria.
    os.chdir("/tmp")
%pip install -q duckdb pyarrow seaborn plotly
import duckdb, seaborn, plotly
print("DuckDB", duckdb.__version__, "· Seaborn", seaborn.__version__,
      "· Plotly", plotly.__version__)

## 0. Datos y plantilla

El generador de la unidad 9 (50 máquinas) y la plantilla `estilo_empresa` del apartado 10.3.

In [ ]:
import numpy as np
import pandas as pd

def generar_lecturas(maquinas=50, horas=24 * 365, semilla=0):
    """Lecturas horarias simuladas de una flota durante un año."""
    rng = np.random.default_rng(semilla)
    ritmo = rng.uniform(0.5, 1.5, maquinas)      # desgaste por hora
    desgaste = rng.uniform(0, 300, maquinas)     # desde la revisión
    filas = []
    for h in range(horas):
        desgaste += ritmo
        temp = 60 + 0.01 * desgaste + rng.normal(0, 2, maquinas)
        if h >= 24 * 273:              # desde octubre, media flota
            temp[: maquinas // 2] += 3  # con sensor descalibrado
        vib = 2 + 0.004 * desgaste + rng.normal(0, 0.5, maquinas)
        kw = 40 + 0.5 * (temp - 60) + rng.normal(0, 3, maquinas)
        z = -9 + 0.006 * desgaste + 0.8 * (vib - 2)
        averia = rng.random(maquinas) < 1 / (1 + np.exp(-z))
        filas.append((np.full(maquinas, h), np.arange(maquinas),
                      temp, vib, kw, desgaste.copy(), averia))
        desgaste[averia] = 0                      # se repara
    h, m, t, v, k, d, a = map(np.concatenate, zip(*filas))
    df = pd.DataFrame({
        "fecha": pd.Timestamp("2025-01-01") + pd.to_timedelta(h, "h"),
        "maquina": m, "temperatura": t, "vibracion": v,
        "consumo_kw": k, "horas_desde_revision": d, "averia": a})
    # objetivo: ¿se averiará la máquina en las próximas 24 horas?
    df = df.sort_values(["maquina", "fecha"], ignore_index=True)
    def en_24h(s):                               # ¿avería en las 24 h
        futuro = s[::-1].rolling(24, 1).max()[::-1]   # siguientes?
        return futuro.shift(-1, fill_value=0)
    proxima = df.groupby("maquina")["averia"].transform(en_24h)
    df["averia_24h"] = proxima.astype(bool)
    return df

In [ ]:
lecturas = generar_lecturas()
lecturas = lecturas.astype({"maquina": "int16", "temperatura": "float32",
    "vibracion": "float32", "consumo_kw": "float32",
    "horas_desde_revision": "float32"})
lecturas.to_parquet("lecturas.parquet", index=False)
print(len(lecturas), "lecturas guardadas en lecturas.parquet")

In [ ]:
import matplotlib as mpl
from cycler import cycler

def estilo_empresa():
    """Aspecto común para todos los gráficos de la empresa."""
    mpl.rcParams.update({
        "font.family": "sans-serif", "font.size": 7,
        "figure.figsize": (4.4, 2.6),          # 11 cm de ancho
        "axes.prop_cycle": cycler(color=[
            "#1F5FA8", "#E08A1E", "#7FA7D6", "#14304F", "#9AA5B1"]),
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": "#E3E8EE",
        "lines.linewidth": 1.2, "lines.markersize": 3.5,
        "grid.linewidth": 0.6, "axes.axisbelow": True,
        "axes.titlesize": 7.5, "axes.titleweight": "bold",
        "axes.titlelocation": "left", "legend.frameon": False,
        "savefig.bbox": "tight", "pdf.fonttype": 42})

## 1. Tarta, barras y puntos (ejemplo 1)

In [ ]:
import duckdb
import matplotlib.pyplot as plt

estilo_empresa()

averias = duckdb.sql("""
    SELECT month(fecha) AS mes, count_if(averia)::INTEGER AS n
    FROM 'lecturas.parquet' GROUP BY mes ORDER BY mes
""").df()
meses = ["E", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]
pares = range(0, 12, 2)                    # una etiqueta de cada dos

fig, (a, b, c) = plt.subplots(1, 3, figsize=(4.4, 1.7))
a.pie(averias["n"], labels=meses, startangle=90,
      counterclock=False)
a.set_title("Tarta")
b.bar(range(12), averias["n"])
b.set_title("Barras")
c.plot(range(12), averias["n"], "o")
c.set_title("Puntos, eje ajustado")
for eje in (b, c):
    eje.set_xticks(pares, [meses[i] for i in pares])
fig.tight_layout()
fig.savefig("tarta_barras.pdf")
print(averias["n"].tolist())

**Prueba.** Ordena las barras de mayor a menor. ¿Qué pregunta responde ahora el gráfico?

## 2. Paneles con eje compartido

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

estilo_empresa()
m = duckdb.sql("""
    SELECT fecha, temperatura, vibracion, horas_desde_revision,
           averia
    FROM 'lecturas.parquet'
    WHERE maquina = 10 AND month(fecha) = 5
    ORDER BY fecha
""").df()

columnas = [("temperatura", "Temperatura (°C)"),
            ("vibracion", "Vibración"),
            ("horas_desde_revision", "Horas desde revisión")]
fig, ejes = plt.subplots(3, 1, figsize=(4.4, 3.4), sharex=True)
for eje, (col, nombre) in zip(ejes, columnas):
    eje.plot(m["fecha"], m[col], lw=0.5)
    eje.set_ylabel(nombre)
    for f in m.loc[m["averia"], "fecha"]:        # cada avería
        eje.axvline(f, color="C1", lw=1, zorder=3)
ejes[2].xaxis.set_major_formatter(mdates.DateFormatter("%d"))
ejes[2].set_xlabel("Día de mayo")
primera = m.loc[m["averia"], "fecha"].iloc[0]
ejes[2].annotate("avería: el contador vuelve a 0",
                 xy=(primera, 0), xytext=(15, 25),
                 textcoords="offset points",
                 arrowprops={"arrowstyle": "->"})
ejes[0].set_title("Máquina 10 en mayo")
fig.align_ylabels()
fig.savefig("panel_maquina.pdf")
print(len(m), "lecturas,", int(m["averia"].sum()), "averías")

**Prueba.** Cambia el número de máquina y el mes en la consulta.

## 3. Escalas de color (ejemplo 2)

In [ ]:
import duckdb
import numpy as np
import matplotlib.pyplot as plt

estilo_empresa()
t = duckdb.sql("""
    SELECT maquina, month(fecha) AS mes, avg(temperatura) AS t
    FROM 'lecturas.parquet' GROUP BY ALL
""").df().pivot(index="maquina", columns="mes", values="t")

def luminancia(nombre, n=256):
    """Luminancia relativa de cada color (0 negro, 1 blanco)."""
    rgb = plt.get_cmap(nombre)(np.linspace(0, 1, n))[:, :3]
    lin = np.where(rgb <= 0.04045, rgb / 12.92,
                   ((rgb + 0.055) / 1.055) ** 2.4)
    return lin @ [0.2126, 0.7152, 0.0722]

fig, ejes = plt.subplot_mosaic([["jet", "viridis"], ["lum", "lum"]],
                               figsize=(4.4, 3.8))
for mapa in ["jet", "viridis"]:
    eje = ejes[mapa]
    im = eje.imshow(t, aspect="auto", cmap=mapa,
                    extent=[0.5, 12.5, 199.5, -0.5])
    eje.set(title=mapa, xlabel="Mes", ylabel="Máquina",
            xticks=[1, 4, 7, 10, 12])
    eje.grid(False)
    fig.colorbar(im, ax=eje, label="°C", shrink=0.9)
for mapa in ["jet", "viridis"]:
    y = luminancia(mapa)
    s = np.sign(np.diff(y))
    s = s[s != 0]                        # ignora tramos planos
    giros = int(np.sum(s[1:] != s[:-1]))
    ejes["lum"].plot(np.linspace(0, 1, len(y)), y,
                 label=f"{mapa} ({giros} cambios de sentido)")
    print(mapa, "cambios de sentido de la luminancia:", giros)
ejes["lum"].set(title="Luminancia a lo largo del mapa",
            xlabel="Posición en la escala", ylabel="Luminancia")
ejes["lum"].legend(loc="upper left")
fig.tight_layout(h_pad=1.5)
fig.savefig("color.pdf")

**Prueba.** Añade `"cividis"` y `"coolwarm"` a la comparación de luminancias. ¿Cuál es divergente?

## 4. Seaborn (ejemplo 3)

In [ ]:
import time
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

estilo_empresa()
d = pd.read_parquet("lecturas.parquet")
d["mes"] = d["fecha"].dt.month
d["grupo"] = d["maquina"].map(lambda m: "afectada" if m < 25
                              else "sin cambio")
d["tramo_horas"] = (d["horas_desde_revision"] // 50) * 50 + 25
d["aviso"] = d["averia_24h"].map({True: "avería en 24 h",
                                  False: "sin avería"})

t0 = time.perf_counter()
fig, ejes = plt.subplot_mosaic([[0, 2], [1, 1]],
                               figsize=(4.4, 4.0))
sns.histplot(d, x="vibracion", hue="aviso", stat="density",
             common_norm=False, element="step", bins=60, ax=ejes[0])
sns.boxplot(d[d["mes"] >= 7], x="mes", y="temperatura",
            hue="grupo", fliersize=0, ax=ejes[1])
sns.lineplot(d[d["tramo_horas"] < 600], x="tramo_horas",
             y="averia_24h", errorbar=("se", 2), marker="o",
             ax=ejes[2])
ejes[0].set(title="Vibración según haya avería", ylim=(0, 0.7),
            xlabel="Vibración", ylabel="Densidad")
ejes[1].set(title="Temperatura de julio a diciembre",
            xlabel="Mes", ylabel="Temperatura (°C)", ylim=(54, 80))
sns.move_legend(ejes[1], "upper left", ncol=2, title=None)
sns.move_legend(ejes[0], "upper right", title=None)
ejes[2].set(title="Proporción de avería en 24 h",
            xlabel="Horas desde la revisión", ylabel="Proporción")
fig.tight_layout()
fig.savefig("seaborn.pdf")
print(f"{time.perf_counter() - t0:.1f} s")
print(d.groupby(["grupo", d["mes"] >= 10])["temperatura"]
       .median().round(1))
print(d.groupby("tramo_horas")["averia_24h"].mean()
       .loc[[25, 225, 425, 575]].round(3))

**Ejercicio.** Dibuja con `sns.histplot` la temperatura de octubre con `hue="grupo"`.

> Cleveland, W. S. y McGill, R. (1984). Graphical perception. *JASA*, 79(387), 531–554. https://doi.org/10.1080/01621459.1984.10478080
>
> Crameri, F., Shephard, G. E. y Heron, P. J. (2020). The misuse of colour in science communication. *Nature Communications*, 11, 5444. https://doi.org/10.1038/s41467-020-19160-7
>
> Waskom, M. L. (2021). seaborn: statistical data visualization. *JOSS*, 6(60), 3021. https://doi.org/10.21105/joss.03021